In [1]:
#!pip install https://nexus.cnj.jus.br/repository/pypi-internal/packages/codex-cliente/3.282.0/codex-cliente-3.282.0.tar.gz

In [2]:
from __future__ import annotations
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any
import yaml
from typing import Any, Callable
from collections.abc import Iterable
import json

from codex_cliente.codex_cliente import CodexCliente
from magma_cliente import MagmaCliente



In [3]:
def carregar_configuracao(caminho: str = "configuracao.yml") -> dict:
    arq = Path(caminho)
    if not arq.exists():
        raise FileNotFoundError(f"Arquivo de configuração não encontrado: {caminho}")
    return yaml.safe_load(arq.read_text(encoding="UTF-8"))

In [4]:
# ── Lista de origemIds a comparar ────────────────────────────────────────────
LISTA_ORIGEM_IDS = [
    "154256",
    "154261",
    "154262",
]

In [5]:
# ============================================================
# DE-PARA DE CAMPOS COMUNS
# ============================================================

ROOT_FIELD_SPECS = [
    {"label": "Nome Fonte de Dados", "rudis": "fonteDados.nome", "codex": "fonteDados.nome"},
    {"label": "Origem ID", "rudis": "origemId", "codex": "origemId"},
    {"label": "Número", "rudis": "numero", "codex": "numero"},
    {"label": "Número Histórico", "rudis": "numeroHistorico", "codex": "numeroHistorico"},
    {"label": "Instância", "rudis": "instancia", "codex": "instancia"},
    {"label": "Data Ajuizamento", "rudis": "dataAjuizamento", "codex": "dataAjuizamento"},
    {"label": "Justiça Gratuita", "rudis": "justicaGratuita", "codex": "justicaGratuita"},
    {"label": "Nível Sigilo", "rudis": "nivelSigilo", "codex": "nivelSigilo"},
    {"label": "Liminar", "rudis": "liminar", "codex": "liminar"},
    {"label": "Valor Causa", "rudis": "valorCausa", "codex": "valorCausa"},
    {"label": "Tipo Justiça", "rudis": "tipoJustica", "codex": "tipoJustica"},
    {"label": "Natureza", "rudis": "natureza", "codex": "natureza"},
    {"label": "Situação Migração", "rudis": "situacaoMigracao", "codex": "situacaoMigracao"},
    {"label": "Última Sync Metadados", "rudis": "dataUltimaSincronizacaoMetadados", "codex": "dataUltimaSincronizacaoMetadados"},
    {"label": "Última Sync Documentos", "rudis": "dataUltimaSincronizacaoDocumentos", "codex": "dataUltimaSincronizacaoDocumentos"},
    {"label": "Última Sync Movimentos", "rudis": "dataUltimaSincronizacaoMovimentos", "codex": "dataUltimaSincronizacaoMovimentos"},
    {"label": "Última Sync Partes", "rudis": "dataUltimaSincronizacaoPartes", "codex": "dataUltimaSincronizacaoPartes"},
    {"label": "Última Compilação", "rudis": "dataUltimaCompilacao", "codex": "dataUltimaCompilacao"},
]

COLLECTION_SPECS = [
    {
        "label": "Assuntos",
        "rudis_path": "assuntos",
        "codex_path": "assuntos",
        "key": "origemId",
        "fields": [
            {"label": "ID vínculo", "rudis": "id", "codex": "id"},
            {"label": "Origem ID", "rudis": "origemId", "codex": "origemId"},
            {"label": "Assunto ID", "rudis": "assunto.id", "codex": "assunto.id"},
            {"label": "Assunto Recebido ID", "rudis": "assuntoRecebidoId", "codex": "assuntoRecebidoId"},
            {"label": "Assunto Local", "rudis": "assuntoLocal", "codex": "assuntoLocal"},
            {"label": "Data Exclusão", "rudis": "dataExclusao", "codex": "dataExclusao"},
        ],
    },
    {
        "label": "Documentos",
        "rudis_path": "documentos",
        "codex_path": "documentos",
        "key": "origemId",
        "fields": [
            {"label": "ID documento", "rudis": "id", "codex": "id"},
            {"label": "Origem ID", "rudis": "origemId", "codex": "origemId"},
            {"label": "Nome", "rudis": "nome", "codex": "nome"},
            {"label": "Nível Sigilo", "rudis": "nivelSigilo", "codex": "nivelSigilo"},
            {"label": "Data Juntada", "rudis": "dataJuntada", "codex": "dataJuntada"},
            {"label": "Tipo Local ID", "rudis": "tipoLocal.id", "codex": "tipoLocal.id"},
            {"label": "Tipo Local Origem", "rudis": "tipoLocal.origemId", "codex": "tipoLocal.origemId"},
            {"label": "Tipo Local Nome", "rudis": "tipoLocal.nome", "codex": "tipoLocal.nome"},
            {"label": "Processado", "rudis": "processado", "codex": "processado"},
            {"label": "Tipo Arquivo", "rudis": "tipoArquivo", "codex": "tipoArquivo"},
            {"label": "Categoria Arquivo", "rudis": "categoriaArquivo", "codex": "categoriaArquivo"},
            {"label": "Data Exclusão", "rudis": "dataExclusao", "codex": "dataExclusao"},
            {"label": "Pai ID", "rudis": "pai.id", "codex": "pai.id"},
            {"label": "Arquivo ID", "rudis": "arquivo.id", "codex": "arquivo.id"},
            {"label": "Arquivo Origem ID", "rudis": "arquivo.origemId", "codex": "arquivo.origemId"},
            {"label": "Hash", "rudis": "arquivo.hash", "codex": "arquivo.hash"},
            {"label": "Tamanho", "rudis": "arquivo.tamanho", "codex": "arquivo.tamanho"},
            {"label": "Tamanho Texto", "rudis": "arquivo.tamanhoTexto", "codex": "arquivo.tamanhoTexto"},
            {"label": "Arquivo Tipo", "rudis": "arquivo.tipo", "codex": "arquivo.tipo"},
            {"label": "Arquivo Categoria", "rudis": "arquivo.categoria", "codex": "arquivo.categoria"},
            {"label": "Transformações", "rudis": "arquivo.transformacoes", "codex": "arquivo.transformacoes"},
            {"label": "Data Processamento Arquivo", "rudis": "arquivo.dataProcessamento", "codex": "arquivo.dataProcessamento"},
            {"label": "Tempo Processamento", "rudis": "arquivo.tempoProcessamento", "codex": "arquivo.tempoProcessamento"},
            {"label": "Tempo Conversão", "rudis": "arquivo.tempoConversao", "codex": "arquivo.tempoConversao"},
            {"label": "Versão Prisma", "rudis": "arquivo.versaoPrisma", "codex": "arquivo.versaoPrisma"},
            {"label": "Qtd Páginas", "rudis": "arquivo.quantidadePaginas", "codex": "arquivo.quantidadePaginas"},
            {"label": "Qtd Imagens", "rudis": "arquivo.quantidadeImagens", "codex": "arquivo.quantidadeImagens"},
            {"label": "Data Processamento Estrutural", "rudis": "dataProcessamento", "codex": "dataProcessamentoArquivoEstrutura"},
            {"label": "Signatários IDs", "rudis": "__signatarios_ids__", "codex": "__signatarios_ids__"},
        ],
    },
    {
        "label": "Histórico Classes",
        "rudis_path": "historicoClasses",
        "codex_path": "historicoClasses",
        "key": "origemId",
        "fields": [
            {"label": "ID histórico", "rudis": "id", "codex": "id"},
            {"label": "Origem ID", "rudis": "origemId", "codex": "origemId"},
            {"label": "Classe ID", "rudis": "classe.id", "codex": "classe.id"},
            {"label": "Data Início", "rudis": "dataInicio", "codex": "dataInicio"},
            {"label": "Data Fim", "rudis": "dataFim", "codex": "dataFim"},
            {"label": "Data Exclusão", "rudis": "dataExclusao", "codex": "dataExclusao"},
            {"label": "Classe Recebida ID", "rudis": "classeRecebidaId", "codex": "classeRecebidaId"},
        ],
    },
    {
        "label": "Histórico Distribuições",
        "rudis_path": "historicoDistribuicoes",
        "codex_path": "historicoDistribuicoes",
        "key": "origemId",
        "fields": [
            {"label": "ID histórico", "rudis": "id", "codex": "id"},
            {"label": "Origem ID", "rudis": "origemId", "codex": "origemId"},
            {"label": "Data Início", "rudis": "dataInicio", "codex": "dataInicio"},
            {"label": "Data Fim", "rudis": "dataFim", "codex": "dataFim"},
            {"label": "Data Exclusão", "rudis": "dataExclusao", "codex": "dataExclusao"},
            {"label": "Órgão Julgador ID", "rudis": "orgaoJulgador.id", "codex": "orgaoJulgador.id"},
            {"label": "Órgão Julgador Nome", "rudis": "orgaoJulgador.nome", "codex": "orgaoJulgador.nome"},
            {"label": "Órgão Julgador Instância", "rudis": "orgaoJulgador.instancia", "codex": "orgaoJulgador.instancia"},
            {"label": "Órgão Julgador Situação", "rudis": "orgaoJulgador.situacao", "codex": "orgaoJulgador.situacao"},
            {"label": "Localidade Nome", "rudis": "orgaoJulgador.localidade.nome", "codex": "orgaoJulgador.localidade.nome"},
            {"label": "Órgão Julgador Local ID", "rudis": "orgaoJulgadorLocal.id", "codex": "orgaoJulgadorLocal.id"},
            {"label": "Órgão Julgador Local Origem", "rudis": "orgaoJulgadorLocal.origemId", "codex": "orgaoJulgadorLocal.origemId"},
            {"label": "Órgão Julgador Local Nome", "rudis": "orgaoJulgadorLocal.nome", "codex": "orgaoJulgadorLocal.nome"},
            {"label": "Órgão Julgador Local Situação", "rudis": "orgaoJulgadorLocal.situacao", "codex": "orgaoJulgadorLocal.situacao"},
            {"label": "Jurisdição Local ID", "rudis": "orgaoJulgadorLocal.jurisdicaoLocal.id", "codex": "orgaoJulgadorLocal.jurisdicaoLocal.id"},
            {"label": "Jurisdição Local Origem", "rudis": "orgaoJulgadorLocal.jurisdicaoLocal.origemId", "codex": "orgaoJulgadorLocal.jurisdicaoLocal.origemId"},
            {"label": "Jurisdição Local Descrição", "rudis": "orgaoJulgadorLocal.jurisdicaoLocal.descricao", "codex": "orgaoJulgadorLocal.jurisdicaoLocal.descricao"},
        ],
    },
    {
        "label": "Movimentos",
        "rudis_path": "movimentos",
        "codex_path": "movimentos",
        "key": "origemId",
        "fields": [
            {"label": "ID movimento", "rudis": "id", "codex": "id"},
            {"label": "Origem ID", "rudis": "origemId", "codex": "origemId"},
            {"label": "Data", "rudis": "data", "codex": "data"},
            {"label": "Descrição", "rudis": "descricao", "codex": "descricao"},
            {"label": "Data Exclusão", "rudis": "dataExclusao", "codex": "dataExclusao"},
            {"label": "Tipo ID", "rudis": "tipo.id", "codex": "tipo.id"},
            {"label": "Tipo Nome", "rudis": "tipo.nome", "codex": "tipo.nome"},
            {"label": "Tipo Descrição", "rudis": "tipo.descricao", "codex": "tipo.descricao"},
            {"label": "Tipo Hierarquia", "rudis": "tipo.hierarquia", "codex": "tipo.hierarquia"},
            {"label": "Tipo Movimento Recebido ID", "rudis": "tipoMovimentoRecebidoId", "codex": "tipoMovimentoRecebidoId"},
            {"label": "Tipo Local ID", "rudis": "tipoLocal.id", "codex": "tipoLocal.id"},
            {"label": "Tipo Local Origem", "rudis": "tipoLocal.origemId", "codex": "tipoLocal.origemId"},
            {"label": "Tipo Local Nome", "rudis": "tipoLocal.nome", "codex": "tipoLocal.nome"},
            {"label": "Documento ID", "rudis": "documento.id", "codex": "documento.id"},
            {"label": "Documento Nome", "rudis": "documento.nome", "codex": "documento.nome"},
            {"label": "Documento Nível Sigilo", "rudis": "documento.nivelSigilo", "codex": "documento.nivelSigilo"},
            {"label": "Documento Processado", "rudis": "documento.processado", "codex": "documento.processado"},
            {"label": "Usuário Login", "rudis": "usuarioLogin", "codex": "usuarioLogin"},
            {"label": "Complementos", "rudis": "__complementos__", "codex": "__complementos__"},
        ],
    },
    {
        "label": "Partes",
        "rudis_path": "partes",
        "codex_path": "partes",
        "key": "origemId",
        "fields": [
            {"label": "ID parte", "rudis": "id", "codex": "id"},
            {"label": "Origem ID", "rudis": "origemId", "codex": "origemId"},
            {"label": "Pessoa ID", "rudis": "pessoa.id", "codex": "pessoa.id"},
            {"label": "Sigilosa", "rudis": "sigilosa", "codex": "sigilosa"},
            {"label": "Situação", "rudis": "situacao", "codex": "situacao"},
            {"label": "Polo", "rudis": "polo", "codex": "polo"},
            {"label": "Qtd Relacionamentos", "rudis": "__len_relacionamentos__", "codex": "__len_relacionamentos__"},
            {"label": "Representantes", "rudis": "__representantes__", "codex": "__representantes__"},
        ],
    },
]


In [6]:
# ============================================================
# MODELOS DE RESULTADO
# ============================================================

@dataclass
class ResultadoCampo:
    bloco: str
    identificador: str
    campo: str
    valor_rudis: Any
    valor_codex: Any

    @property
    def igual(self) -> bool:
        return normalizar_valor(self.valor_rudis) == normalizar_valor(self.valor_codex)

    @property
    def status(self) -> str:
        return "✅" if self.igual else "⚠️"


@dataclass
class ResultadoProcesso:
    origem_id: str
    numero: str
    campos: list[ResultadoCampo] = field(default_factory=list)
    erro: str | None = None

    @property
    def sincronizado(self) -> bool:
        return self.erro is None and all(c.igual for c in self.campos)

    @property
    def qtd_divergencias(self) -> int:
        return sum(1 for c in self.campos if not c.igual)


# ============================================================
# HELPERS
# ============================================================

_MISSING = object()


def safe_get(obj: Any, path: str, default: Any = None) -> Any:
    if path.startswith("__") and path.endswith("__"):
        return extrair_campo_virtual(obj, path, default)

    atual = obj
    for parte in path.split("."):
        if atual is None:
            return default
        if isinstance(atual, dict):
            atual = atual.get(parte, default)
        else:
            return default
    return atual


def extrair_campo_virtual(obj: dict, path: str, default: Any = None) -> Any:
    if path == "__signatarios_ids__":
        signatarios = obj.get("signatarios") or []
        ids = []
        for item in signatarios:
            if not isinstance(item, dict):
                continue
            if "signatario" in item and isinstance(item["signatario"], dict):
                ids.append(item["signatario"].get("id"))
            else:
                ids.append(item.get("id"))
        return sorted(v for v in ids if v is not None)

    if path == "__complementos__":
        complementos = obj.get("complementos") or []
        canonicos = []
        for c in complementos:
            canonicos.append({
                "origemId": c.get("origemId"),
                "tipo.id": safe_get(c, "tipo.id"),
                "tipo.descricao": safe_get(c, "tipo.descricao"),
                "valor": c.get("valor"),
                "tabelado.id": safe_get(c, "tabelado.id"),
                "tabelado.descricao": safe_get(c, "tabelado.descricao"),
            })
        return sorted(canonicos, key=lambda x: json.dumps(normalizar_valor(x), ensure_ascii=False, sort_keys=True))

    if path == "__len_relacionamentos__":
        return len(obj.get("relacionamentos") or [])

    if path == "__representantes__":
        reps = obj.get("representantes") or []
        canonicos = []
        for r in reps:
            canonicos.append({
                "id": r.get("id"),
                "origemId": r.get("origemId"),
                "tipo": r.get("tipo"),
                "representante.id": safe_get(r, "representante.id"),
                "enviarIntimacao": r.get("enviarIntimacao"),
                "dataExclusao": r.get("dataExclusao"),
            })
        return sorted(canonicos, key=lambda x: json.dumps(normalizar_valor(x), ensure_ascii=False, sort_keys=True))

    return default


def normalizar_valor(valor: Any) -> Any:
    if isinstance(valor, dict):
        return {k: normalizar_valor(v) for k, v in sorted(valor.items())}
    if isinstance(valor, list):
        return [normalizar_valor(v) for v in valor]
    return valor


def indexar_colecao(itens: list[dict], key_path: str) -> dict[str, dict]:
    indice = {}
    for item in itens or []:
        chave = safe_get(item, key_path, None)
        if chave is None:
            continue
        indice[str(chave)] = item
    return indice


# ============================================================
# COMPARAÇÃO
# ============================================================

def comparar_campos_raiz(processo_rudis: dict, processo_codex: dict) -> list[ResultadoCampo]:
    resultados = []

    for spec in ROOT_FIELD_SPECS:
        resultados.append(
            ResultadoCampo(
                bloco="Processo",
                identificador=processo_rudis.get("origemId", "?"),
                campo=spec["label"],
                valor_rudis=safe_get(processo_rudis, spec["rudis"]),
                valor_codex=safe_get(processo_codex, spec["codex"]),
            )
        )

    # contagens gerais por coleção
    for spec in COLLECTION_SPECS:
        nome_bloco = spec["label"]
        qtd_rudis = len(safe_get(processo_rudis, spec["rudis_path"], []) or [])
        qtd_codex = len(safe_get(processo_codex, spec["codex_path"], []) or [])

        resultados.append(
            ResultadoCampo(
                bloco="Resumo Coleções",
                identificador=nome_bloco,
                campo="Quantidade",
                valor_rudis=qtd_rudis,
                valor_codex=qtd_codex,
            )
        )

    return resultados


def comparar_colecao(
    processo_rudis: dict,
    processo_codex: dict,
    spec: dict,
) -> list[ResultadoCampo]:
    bloco = spec["label"]
    itens_rudis = safe_get(processo_rudis, spec["rudis_path"], []) or []
    itens_codex = safe_get(processo_codex, spec["codex_path"], []) or []

    idx_rudis = indexar_colecao(itens_rudis, spec["key"])
    idx_codex = indexar_colecao(itens_codex, spec["key"])

    chaves = sorted(set(idx_rudis.keys()) | set(idx_codex.keys()))
    resultados: list[ResultadoCampo] = []

    for chave in chaves:
        item_rudis = idx_rudis.get(chave)
        item_codex = idx_codex.get(chave)

        if item_rudis is None:
            resultados.append(
                ResultadoCampo(
                    bloco=bloco,
                    identificador=chave,
                    campo="[ITEM]",
                    valor_rudis="[AUSENTE]",
                    valor_codex="[PRESENTE]",
                )
            )
            continue

        if item_codex is None:
            resultados.append(
                ResultadoCampo(
                    bloco=bloco,
                    identificador=chave,
                    campo="[ITEM]",
                    valor_rudis="[PRESENTE]",
                    valor_codex="[AUSENTE]",
                )
            )
            continue

        for field_spec in spec["fields"]:
            resultados.append(
                ResultadoCampo(
                    bloco=bloco,
                    identificador=chave,
                    campo=field_spec["label"],
                    valor_rudis=safe_get(item_rudis, field_spec["rudis"]),
                    valor_codex=safe_get(item_codex, field_spec["codex"]),
                )
            )

    return resultados


def comparar_processo(processo_rudis: dict, processo_codex: dict) -> list[ResultadoCampo]:
    resultados = []
    resultados.extend(comparar_campos_raiz(processo_rudis, processo_codex))

    for spec in COLLECTION_SPECS:
        resultados.extend(comparar_colecao(processo_rudis, processo_codex, spec))

    return resultados


def buscar_e_comparar(
    origem_id: str,
    rudis_cliente,
    codex_cliente,
) -> ResultadoProcesso:
    try:
        processo_rudis = rudis_cliente.get(
            f"processos/compilados/TJES_PJEPG/{origem_id}"
        ).json()

        processos_codex = codex_cliente.processo_compilado.recuperar_por_numero(
            processo_rudis["numero"]
        )

        processo_codex = next(
            (p for p in processos_codex if str(p.get("origemId")) == str(origem_id)),
            None,
        )

        if processo_codex is None:
            return ResultadoProcesso(
                origem_id=origem_id,
                numero=processo_rudis.get("numero", "?"),
                erro=f"origemId={origem_id} não encontrado no CODEX",
            )

        campos = comparar_processo(processo_rudis, processo_codex)

        return ResultadoProcesso(
            origem_id=origem_id,
            numero=processo_rudis.get("numero", "?"),
            campos=campos,
        )

    except Exception as exc:  # noqa: BLE001
        return ResultadoProcesso(
            origem_id=origem_id,
            numero="?",
            erro=str(exc),
        )


# ============================================================
# IMPRESSÃO
# ============================================================

COL_BLOCO = 24
COL_ITEM = 18
COL_CAMPO = 32
COL_VALOR = 28


def _linha_separador() -> str:
    return "-" * (COL_BLOCO + COL_ITEM + COL_CAMPO + COL_VALOR * 2 + 24)


def _cabecalho() -> None:
    print(
        f"| {'Bloco':<{COL_BLOCO}}"
        f" | {'Item':<{COL_ITEM}}"
        f" | {'Campo':<{COL_CAMPO}}"
        f" | {'RUDIS':<{COL_VALOR}}"
        f" | {'CODEX':<{COL_VALOR}}"
        f" | St |"
    )
    print(_linha_separador())


def _to_str(valor: Any) -> str:
    if isinstance(valor, (list, dict)):
        txt = json.dumps(valor, ensure_ascii=False, sort_keys=True)
    else:
        txt = str(valor)
    return txt[:COL_VALOR - 1] + "…" if len(txt) > COL_VALOR else txt


def imprimir_resultado(res: ResultadoProcesso) -> None:
    print()
    print(f"PROCESSO: {res.numero} | origemId={res.origem_id}")
    print(_linha_separador())

    if res.erro:
        print(f"| {'[ERRO]':<{COL_BLOCO}} | {'':<{COL_ITEM}} | {'':<{COL_CAMPO}} | {res.erro:<{COL_VALOR * 2 + 3}} | ❌ |")
        print(_linha_separador())
        return

    _cabecalho()
    for campo in res.campos:
        print(
            f"| {campo.bloco:<{COL_BLOCO}}"
            f" | {campo.identificador:<{COL_ITEM}}"
            f" | {campo.campo:<{COL_CAMPO}}"
            f" | {_to_str(campo.valor_rudis):<{COL_VALOR}}"
            f" | {_to_str(campo.valor_codex):<{COL_VALOR}}"
            f" | {campo.status} |"
        )

    status_processo = (
        "✅ Sincronizado"
        if res.sincronizado
        else f"⚠️ {res.qtd_divergencias} divergência(s)"
    )
    print(_linha_separador())
    print(status_processo)
    print(_linha_separador())


def imprimir_resumo(resultados: list[ResultadoProcesso]) -> None:
    total = len(resultados)
    sincronizados = sum(1 for r in resultados if r.sincronizado)
    com_erro = sum(1 for r in resultados if r.erro)
    divergentes = total - sincronizados - com_erro

    print()
    print("═" * 72)
    print(f"RESUMO GERAL — {total} processo(s) analisado(s)")
    print("═" * 72)
    print(f"✅ Sincronizados : {sincronizados}")
    print(f"⚠️ Divergentes   : {divergentes}")
    print(f"❌ Com erro      : {com_erro}")
    print("═" * 72)

In [7]:
def main_lote(lista_ids: list[str]) -> list[ResultadoProcesso]:
    cfg = carregar_configuracao()

    codex_cliente = CodexCliente(
        cfg["codex"]["server"],
        cfg["codex"]["usuario"],
        cfg["codex"]["senha"],
    )
    rudis_cliente = MagmaCliente(
        cfg["rudis"]["server"],
        cfg["rudis"]["usuario"],
        cfg["rudis"]["senha"],
    )

    resultados: list[ResultadoProcesso] = []

    for origem_id in lista_ids:
        res = buscar_e_comparar(origem_id, rudis_cliente, codex_cliente)
        resultados.append(res)
        imprimir_resultado(res)

    imprimir_resumo(resultados)
    return resultados

In [8]:
resultados = main_lote(LISTA_ORIGEM_IDS)


PROCESSO: ? | origemId=154256
----------------------------------------------------------------------------------------------------------------------------------------------------------
| [ERRO]                   |                    |                                  | Could not find a suitable TLS CA certificate bundle, invalid path: C:\Users\RFSCHNEROKE\pip\certificado-filtroweb-2024.crt | ❌ |
----------------------------------------------------------------------------------------------------------------------------------------------------------

PROCESSO: ? | origemId=154261
----------------------------------------------------------------------------------------------------------------------------------------------------------
| [ERRO]                   |                    |                                  | Could not find a suitable TLS CA certificate bundle, invalid path: C:\Users\RFSCHNEROKE\pip\certificado-filtroweb-2024.crt | ❌ |
---------------------------------------------